In [9]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Funciones de ventana").getOrCreate()

In [15]:
df = spark.read.parquet("./data/")

In [16]:
# Es un dataframe bastante sencillo que contiene las columnas nombre, edad y evaluacion

df.show()

+-------+----+------------+----------+
| nombre|edad|departamento|evaluacion|
+-------+----+------------+----------+
| Lazaro|  45|      letras|        98|
|   Raul|  24|  matemática|        76|
|  Maria|  34|  matemática|        27|
|   Jose|  30|     química|        78|
| Susana|  51|     química|        98|
|   Juan|  44|      letras|        89|
|  Julia|  55|      letras|        92|
|  Kadir|  38|arquitectura|        39|
| Lilian|  23|arquitectura|        94|
|   Rosa|  26|      letras|        91|
|   Aian|  50|  matemática|        73|
|Yaneisy|  29|      letras|        89|
|Enrique|  40|     química|        92|
|    Jon|  25|arquitectura|        78|
|  Luisa|  39|arquitectura|        94|
+-------+----+------------+----------+



### Cuales fueron los trabajadores de cada uno de los departamentos con la mayor evaluacion?

In [19]:
# Las funciones de ventana en spark operan en un grupo de filas y devuelven un valor unico para cada fila de entrada

from pyspark.sql.functions import desc, row_number, rank, dense_rank, window, col

In [14]:
from pyspark.sql.window import Window

# crear una especificacion de ventana vamos a necesittar particionar los datos y posteriormente ordenrarlos

windowSpec = Window.partitionBy("departamento").orderBy(desc("evaluacion"))

## Funcion row_number

In [17]:
# se usa para dar el num de filas secuencial comenzando desde 1 hasta el resultado de cada particion de ventana

df.withColumn("row_number", row_number().over(windowSpec)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|row_number|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         2|
|    Jon|  25|arquitectura|        78|         3|
|  Kadir|  38|arquitectura|        39|         4|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Rosa|  26|      letras|        91|         3|
|   Juan|  44|      letras|        89|         4|
|Yaneisy|  29|      letras|        89|         5|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
|  Maria|  34|  matemática|        27|         3|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
|   Jose|  30|     química|        78|         3|
+-------+----+------------+----------+----------+



si observan ya tenemos los trabajadores organizados oparticionados pro departamento porque precisamente esto lo hace la especifiacion de venana y estan ordenados de forma descendiente porque lo indicamos en la ventana

si nos fijamos el depto de arquitectura va de 1 hasta 4  y el de letras de 1 hasta 5 y asi seria el funcionamiento de row_number

In [20]:
df.withColumn("row_number", row_number().over(windowSpec)).filter(col("row_number").isin(1,2)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|row_number|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         2|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
+-------+----+------------+----------+----------+



In [23]:
# Filtramos para obtener solo el trabajador con la mayor evaluación (row_number = 1) en cada departamento
df.withColumn("row_number", row_number().over(windowSpec)).filter(col("row_number").isin(1,2)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|row_number|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         2|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
+-------+----+------------+----------+----------+



In [24]:
# El resultado es que en cada uno de los dto fueron estas dos personas las que resultaron con mejor promedio

## Funcion de ventana rank

la funcion de ventana rank se usa para proporcionar un rango al resultado dentro de una particion de ventana la diferencia es que esta funcion deja huecos en el rango cuando hay empates

In [25]:
df.withColumn("rank", rank().over(windowSpec)).show()

+-------+----+------------+----------+----+
| nombre|edad|departamento|evaluacion|rank|
+-------+----+------------+----------+----+
| Lilian|  23|arquitectura|        94|   1|
|  Luisa|  39|arquitectura|        94|   1|
|    Jon|  25|arquitectura|        78|   3|
|  Kadir|  38|arquitectura|        39|   4|
| Lazaro|  45|      letras|        98|   1|
|  Julia|  55|      letras|        92|   2|
|   Rosa|  26|      letras|        91|   3|
|   Juan|  44|      letras|        89|   4|
|Yaneisy|  29|      letras|        89|   4|
|   Raul|  24|  matemática|        76|   1|
|   Aian|  50|  matemática|        73|   2|
|  Maria|  34|  matemática|        27|   3|
| Susana|  51|     química|        98|   1|
|Enrique|  40|     química|        92|   2|
|   Jose|  30|     química|        78|   3|
+-------+----+------------+----------+----+



In [27]:
# donde huno un empate rank enumero ese empate pero si nos fijamos abajo rank ha saltado al num 3 la numeracion no es consecutiva

## dense_rank

es la misma funcion que rank con la diferencia que esta no deja empates en el dataframe

In [30]:
df.withColumn("dense_rank", dense_rank().over(windowSpec)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|dense_rank|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         1|
|    Jon|  25|arquitectura|        78|         2|
|  Kadir|  38|arquitectura|        39|         3|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Rosa|  26|      letras|        91|         3|
|   Juan|  44|      letras|        89|         4|
|Yaneisy|  29|      letras|        89|         4|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
|  Maria|  34|  matemática|        27|         3|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
|   Jose|  30|     química|        78|         3|
+-------+----+------------+----------+----------+



In [31]:
# cuando hay un empate en la evaluacion los enumero con 1 y despues salta al num 2 y al 3 sucesivamente

## Agregaciones con funciones de venana

resulta que. Cuando trabajamos con funciones de agregacion no es necesario utilizar la clausula orderBy cuando creamos la especificacion de ventana

In [33]:
from pyspark.sql.window import Window

windowSpec_agg = Window.partitionBy("departamento")

In [35]:
from pyspark.sql.functions import min, max, avg

In [37]:
df.withColumn("min", min(col("evaluacion")).over(windowSpec_agg)).withColumn("max", max(col("evaluacion")).over(windowSpec_agg)).withColumn("avg", avg(col("evaluacion")).over(windowSpec_agg)).withColumn("row_number", row_number().over(windowSpec)).show()

+-------+----+------------+----------+---+---+------------------+----------+
| nombre|edad|departamento|evaluacion|min|max|               avg|row_number|
+-------+----+------------+----------+---+---+------------------+----------+
| Lilian|  23|arquitectura|        94| 39| 94|             76.25|         1|
|  Luisa|  39|arquitectura|        94| 39| 94|             76.25|         2|
|    Jon|  25|arquitectura|        78| 39| 94|             76.25|         3|
|  Kadir|  38|arquitectura|        39| 39| 94|             76.25|         4|
| Lazaro|  45|      letras|        98| 89| 98|              91.8|         1|
|  Julia|  55|      letras|        92| 89| 98|              91.8|         2|
|   Rosa|  26|      letras|        91| 89| 98|              91.8|         3|
|   Juan|  44|      letras|        89| 89| 98|              91.8|         4|
|Yaneisy|  29|      letras|        89| 89| 98|              91.8|         5|
|   Raul|  24|  matemática|        76| 27| 76|58.666666666666664|         1|